# RAG 평가셋 검수 및 최종 골든셋 생성

실행 순서: 설정 → 후보 로드·정규화 → 근거 데이터 로드 → ID 검사 → 버튼 검수 → 최종 파일 생성

- REVIEW_MODE = "personal": 현재 프로젝트의 개인 후보를 검수합니다.
- REVIEW_MODE = "team": data/evaluation/team_candidates/candidates_이름.jsonl 파일들을 합쳐 검수합니다.
- reference는 검수된 expected_facts와 필요한 안전 문구를 연결해 자동 생성합니다.
- 기존 검수 결과가 있으면 case_id를 기준으로 이어서 불러옵니다.


In [15]:
import copy
import json
from collections import Counter
from pathlib import Path

# notebooks 폴더와 프로젝트 루트 어디에서 실행해도 경로가 맞도록 설정
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED = ROOT / "data" / "processed"
EVALUATION = ROOT / "data" / "evaluation"

REVIEW_MODE = "personal"  # "personal" 또는 "team"
REVIEWER_NAME = ""        # 팀원이 각자 검수할 때만 입력 (예: "수민")
OVERWRITE_FINAL = False   # 기존 golden_set_final.jsonl 덮어쓰기 여부

PERSONAL_CANDIDATE_PATHS = [
    EVALUATION / "candidates_pending.jsonl",
    EVALUATION / "candidates_pending_nb01.jsonl",
]
TEAM_DIR = EVALUATION / "team_candidates"

if REVIEW_MODE == "personal":
    REVIEWED_PATH = EVALUATION / "candidates_reviewed.jsonl"
elif REVIEW_MODE == "team":
    suffix = f"_{REVIEWER_NAME}" if REVIEWER_NAME.strip() else ""
    REVIEWED_PATH = EVALUATION / f"team_candidates_reviewed{suffix}.jsonl"
else:
    raise ValueError('REVIEW_MODE는 "personal" 또는 "team"이어야 합니다.')

FINAL_PATH = EVALUATION / "golden_set_final.jsonl"
EVALUATION.mkdir(parents=True, exist_ok=True)

print("프로젝트 루트:", ROOT)
print("검수 모드:", REVIEW_MODE)
print("검수 결과:", REVIEWED_PATH)
print("최종 파일:", FINAL_PATH)


프로젝트 루트: c:\Users\Playdata\Desktop\mle-01-p3-team2
검수 모드: personal
검수 결과: c:\Users\Playdata\Desktop\mle-01-p3-team2\data\evaluation\candidates_reviewed.jsonl
최종 파일: c:\Users\Playdata\Desktop\mle-01-p3-team2\data\evaluation\golden_set_final.jsonl


In [16]:
def read_jsonl(path):
    with path.open("r", encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]


def write_jsonl(path, rows, overwrite=False):
    if path.exists() and not overwrite:
        raise FileExistsError(
            f"{path}이 이미 있습니다. 기존 파일을 확인하거나 "
            "OVERWRITE_FINAL=True로 바꾸세요."
        )

    temporary_path = path.with_suffix(path.suffix + ".tmp")
    with temporary_path.open("w", encoding="utf-8") as file:
        for row in rows:
            file.write(json.dumps(row, ensure_ascii=False) + "\n")
    temporary_path.replace(path)


def ensure_list(value):
    if value is None:
        return []
    return value if isinstance(value, list) else [value]


def add_terminal_punctuation(text):
    text = str(text).strip()
    if text and text[-1] not in ".!?。！？":
        text += "."
    return text


def make_reference(row):
    sentences = [
        add_terminal_punctuation(fact)
        for fact in ensure_list(row.get("expected_facts"))
        if str(fact).strip()
    ]

    if row.get("expected_behavior") == "ANSWER_WITH_WARNING":
        sentences.extend(
            add_terminal_punctuation(assertion)
            for assertion in ensure_list(row.get("safety_assertions"))
            if str(assertion).strip()
        )

    # 순서를 유지하면서 완전히 같은 문장만 제거
    return " ".join(dict.fromkeys(sentences))


FIELD_ALIASES = {
    "case_id": ["case_id", "question_id", "id"],
    "question": ["question", "input", "user_input", "query"],
    "query_strategy": ["query_strategy", "strategy", "route", "search_type"],
    "expected_behavior": ["expected_behavior", "behavior"],
    "expected_facts": [
        "expected_facts",
        "ground_truth_facts",
        "reference_facts",
    ],
    "expected_entity_ids": ["expected_entity_ids", "entity_ids"],
    "expected_relation_ids": ["expected_relation_ids", "relation_ids"],
    "expected_evidence_ids": ["expected_evidence_ids", "evidence_ids"],
    "expected_source_chunk_ids": [
        "expected_source_chunk_ids",
        "source_chunk_ids",
        "reference_context_ids",
        "context_ids",
    ],
    "safety_assertions": [
        "safety_assertions",
        "safety_rules",
        "warnings",
    ],
}


def find_value(row, canonical_key, default=None):
    for key in FIELD_ALIASES[canonical_key]:
        if key in row:
            return row[key]
    return default


def normalize_candidate(raw_row, row_number, owner=None):
    row = copy.deepcopy(raw_row)
    warnings = []

    original_case_id = str(
        find_value(row, "case_id", f"ROW_{row_number:04d}")
    )
    question = find_value(row, "question")
    if not question:
        raise KeyError(
            f"{owner or 'personal'} {original_case_id}: "
            "질문 키를 찾을 수 없습니다."
        )

    expected_facts = find_value(row, "expected_facts")
    if expected_facts is None:
        supplied_reference = (
            row.get("reference") or row.get("expected_output")
        )
        if supplied_reference:
            expected_facts = [supplied_reference]
            warnings.append(
                "reference 전체를 기대 사실 1개로 변환함: "
                "원자적 사실로 분리 필요"
            )
        else:
            expected_facts = []
            warnings.append("기대 사실이 없음")

    case_id = (
        f"{owner}__{original_case_id}"
        if owner
        else original_case_id
    )
    normalized = {
        "case_id": case_id,
        "question": str(question).strip(),
        "question_type": row.get(
            "question_type",
            "unclassified",
        ),
        "difficulty": row.get(
            "difficulty",
            "unclassified",
        ),
        "query_strategy": find_value(
            row,
            "query_strategy",
            "unclassified",
        ),
        "expected_behavior": find_value(
            row,
            "expected_behavior",
            "ANSWER",
        ),
        "expected_facts": ensure_list(expected_facts),
        "expected_entity_ids": ensure_list(
            find_value(row, "expected_entity_ids", [])
        ),
        "expected_relation_ids": ensure_list(
            find_value(row, "expected_relation_ids", [])
        ),
        "expected_evidence_ids": ensure_list(
            find_value(row, "expected_evidence_ids", [])
        ),
        "expected_source_chunk_ids": ensure_list(
            find_value(row, "expected_source_chunk_ids", [])
        ),
        "safety_assertions": ensure_list(
            find_value(row, "safety_assertions", [])
        ),
        "status": row.get("status", "PENDING_REVIEW"),
        "review": row.get(
            "review",
            {
                "question_ok": None,
                "facts_ok": None,
                "evidence_ok": None,
                "ids_ok": None,
                "reviewer_note": "",
            },
        ),
        "normalization_warnings": warnings,
    }

    if owner:
        normalized["candidate_owner"] = owner
        normalized["original_case_id"] = original_case_id
    if "generation_metadata" in row:
        normalized["generation_metadata"] = row["generation_metadata"]

    normalized["reference"] = make_reference(normalized)
    return normalized


In [17]:
# 후보 파일 로드 및 공통 스키마 정규화
if REVIEW_MODE == "personal":
    candidate_files = [
        path
        for path in PERSONAL_CANDIDATE_PATHS
        if path.exists()
    ]
else:
    TEAM_DIR.mkdir(parents=True, exist_ok=True)
    candidate_files = sorted(
        TEAM_DIR.glob("candidates_*.jsonl")
    )

if not candidate_files:
    raise FileNotFoundError(
        "검수할 후보 JSONL 파일이 없습니다. "
        "설정 셀의 경로를 확인하세요."
    )

candidates = []
seen_case_ids = set()

for path in candidate_files:
    owner = (
        path.stem.removeprefix("candidates_")
        if REVIEW_MODE == "team"
        else None
    )
    for row_number, raw_row in enumerate(
        read_jsonl(path),
        start=1,
    ):
        row = normalize_candidate(
            raw_row,
            row_number=row_number,
            owner=owner,
        )
        if row["case_id"] in seen_case_ids:
            raise ValueError(
                f"중복 case_id: {row['case_id']}"
            )
        seen_case_ids.add(row["case_id"])
        candidates.append(row)

# 기존 검수 결과를 case_id 기준으로 병합
if REVIEWED_PATH.exists():
    reviewed_by_id = {
        row["case_id"]: row
        for row in read_jsonl(REVIEWED_PATH)
    }
    candidates = [
        reviewed_by_id.get(row["case_id"], row)
        for row in candidates
    ]

# 과거 검수 파일에는 reference가 없을 수 있으므로 다시 생성
for row in candidates:
    row["reference"] = make_reference(row)

print("불러온 파일")
for path in candidate_files:
    print("-", path.name)
print("전체 검수 문항:", len(candidates))
print(
    "상태별 개수:",
    dict(Counter(row.get("status") for row in candidates)),
)

warning_rows = [
    row
    for row in candidates
    if row.get("normalization_warnings")
]
if warning_rows:
    print("\n정규화 확인 필요")
    for row in warning_rows:
        print(
            "-",
            row["case_id"],
            row["normalization_warnings"],
        )


불러온 파일
- candidates_pending.jsonl
전체 검수 문항: 30
상태별 개수: {'PENDING_REVIEW': 30}


In [18]:
# 근거·엔티티·관계·청크 인덱스
evidence_by_id = {
    row["evidence_id"]: row
    for row in read_jsonl(PROCESSED / "evidence.jsonl")
}
entity_by_id = {
    row["entity_id"]: row
    for row in read_jsonl(PROCESSED / "entities.jsonl")
}
relation_by_id = {
    row["relation_id"]: row
    for row in read_jsonl(PROCESSED / "relations.jsonl")
}
chunk_by_id = {
    row["chunk_id"]: row
    for row in read_jsonl(PROCESSED / "chunks.jsonl")
}

print("evidence:", len(evidence_by_id))
print("entity:", len(entity_by_id))
print("relation:", len(relation_by_id))
print("chunk:", len(chunk_by_id))


evidence: 30650
entity: 30933
relation: 46034
chunk: 30661


In [19]:
def validate_ids(candidate):
    errors = []

    for evidence_id in candidate.get(
        "expected_evidence_ids",
        [],
    ):
        if evidence_id not in evidence_by_id:
            errors.append(
                f"존재하지 않는 evidence ID: {evidence_id}"
            )

    for entity_id in candidate.get(
        "expected_entity_ids",
        [],
    ):
        if entity_id not in entity_by_id:
            errors.append(
                f"존재하지 않는 entity ID: {entity_id}"
            )

    for relation_id in candidate.get(
        "expected_relation_ids",
        [],
    ):
        if relation_id not in relation_by_id:
            errors.append(
                f"존재하지 않는 relation ID: {relation_id}"
            )

    expected_evidence = set(
        candidate.get("expected_evidence_ids", [])
    )
    for chunk_id in candidate.get(
        "expected_source_chunk_ids",
        [],
    ):
        chunk = chunk_by_id.get(chunk_id)
        if chunk is None:
            errors.append(
                f"존재하지 않는 chunk ID: {chunk_id}"
            )
            continue
        if chunk.get("evidence_id") not in expected_evidence:
            errors.append(
                f"{chunk_id}가 기대 evidence와 연결되지 않음"
            )

    for relation_id in candidate.get(
        "expected_relation_ids",
        [],
    ):
        relation = relation_by_id.get(relation_id)
        if relation is None:
            continue
        relation_evidence = set(
            relation.get("evidence_ids", [])
        )
        if (
            relation_evidence
            and not relation_evidence & expected_evidence
        ):
            errors.append(
                f"{relation_id}의 근거가 "
                "expected_evidence_ids에 없음"
            )

    return errors


invalid_rows = {
    row["case_id"]: validate_ids(row)
    for row in candidates
}
invalid_rows = {
    case_id: errors
    for case_id, errors in invalid_rows.items()
    if errors
}
print("ID 검사 실패 문항:", len(invalid_rows))
for case_id, errors in invalid_rows.items():
    print("-", case_id, errors)


ID 검사 실패 문항: 0


In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output, display

if not candidates:
    raise ValueError("검수할 문항이 없습니다.")

first_pending = next(
    (
        index
        for index, row in enumerate(candidates)
        if row.get("status") == "PENDING_REVIEW"
    ),
    0,
)
state = {"index": first_pending}

title = widgets.HTML()
question_box = widgets.Textarea(
    disabled=True,
    layout=widgets.Layout(width="100%", height="90px"),
)
facts_box = widgets.Textarea(
    disabled=True,
    layout=widgets.Layout(width="100%", height="140px"),
)
reference_box = widgets.Textarea(
    disabled=True,
    layout=widgets.Layout(width="100%", height="140px"),
)
graph_box = widgets.Textarea(
    disabled=True,
    layout=widgets.Layout(width="100%", height="130px"),
)
evidence_box = widgets.Textarea(
    disabled=True,
    layout=widgets.Layout(width="100%", height="300px"),
)
id_result = widgets.HTML()

toggle_options = [
    ("미검수", None),
    ("통과", True),
    ("수정·제외", False),
]
question_ok = widgets.ToggleButtons(
    options=toggle_options,
    description="질문",
)
facts_ok = widgets.ToggleButtons(
    options=toggle_options,
    description="기대 사실",
)
evidence_ok = widgets.ToggleButtons(
    options=toggle_options,
    description="근거",
)
reviewer_note = widgets.Textarea(
    placeholder="수정 사항이나 제외 이유",
    layout=widgets.Layout(width="100%", height="80px"),
)

previous_button = widgets.Button(description="이전")
approve_button = widgets.Button(
    description="모두 통과",
    button_style="success",
)
save_button = widgets.Button(
    description="현재 판정 저장",
    button_style="info",
)
reject_button = widgets.Button(
    description="제외",
    button_style="danger",
)
next_button = widgets.Button(description="다음")
status_output = widgets.Output()


def save_all():
    temporary_path = REVIEWED_PATH.with_suffix(
        REVIEWED_PATH.suffix + ".tmp"
    )
    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as file:
        for row in candidates:
            row["reference"] = make_reference(row)
            file.write(
                json.dumps(
                    row,
                    ensure_ascii=False,
                )
                + "\n"
            )
    temporary_path.replace(REVIEWED_PATH)


def render():
    row = candidates[state["index"]]
    review = row.get("review", {})
    owner_text = (
        f" · 작성자 {row['candidate_owner']}"
        if row.get("candidate_owner")
        else ""
    )
    title.value = (
        f"<h3>{state['index'] + 1}/{len(candidates)}"
        f"{owner_text} · {row['case_id']} "
        f"· {row['query_strategy']} "
        f"· {row.get('status')}</h3>"
    )
    question_box.value = row["question"]
    facts_box.value = "\n".join(
        f"- {fact}"
        for fact in row.get("expected_facts", [])
    )
    row["reference"] = make_reference(row)
    reference_box.value = row["reference"]

    evidence_texts = []
    for evidence_id in row.get(
        "expected_evidence_ids",
        [],
    ):
        evidence = evidence_by_id.get(evidence_id)
        if evidence is None:
            evidence_texts.append(
                f"[{evidence_id}]\n"
                "근거를 찾을 수 없습니다."
            )
        else:
            evidence_texts.append(
                f"[{evidence_id} / "
                f"{evidence.get('evidence_type', 'unknown')}]\n"
                f"{evidence.get('text', '')}"
            )
    evidence_box.value = (
        "\n\n".join(evidence_texts)
        or "근거 원문 없음"
    )

    graph_lines = []
    for relation_id in row.get(
        "expected_relation_ids",
        [],
    ):
        relation = relation_by_id.get(relation_id)
        if relation is None:
            graph_lines.append(
                f"{relation_id}: 관계 없음"
            )
            continue
        from_entity = entity_by_id.get(
            relation["from_id"],
            {},
        ).get("name", relation["from_id"])
        to_entity = entity_by_id.get(
            relation["to_id"],
            {},
        ).get("name", relation["to_id"])
        graph_lines.append(
            f"{from_entity} "
            f"--{relation['relation_type']}--> "
            f"{to_entity}"
        )
    graph_box.value = (
        "\n".join(graph_lines)
        if graph_lines
        else "그래프 관계 없음"
    )

    errors = validate_ids(row)
    id_result.value = (
        "<b style='color:red'>ID 자동 검사 실패</b><br>"
        + "<br>".join(errors)
        if errors
        else "<b style='color:green'>ID 자동 검사 통과</b>"
    )
    question_ok.value = review.get("question_ok")
    facts_ok.value = review.get("facts_ok")
    evidence_ok.value = review.get("evidence_ok")
    reviewer_note.value = review.get(
        "reviewer_note",
        "",
    )


def save_current(force_reject=False):
    row = candidates[state["index"]]
    id_errors = validate_ids(row)

    q_result = (
        False
        if force_reject
        else question_ok.value
    )
    f_result = (
        False
        if force_reject
        else facts_ok.value
    )
    e_result = (
        False
        if force_reject
        else evidence_ok.value
    )

    row["review"] = {
        "question_ok": q_result,
        "facts_ok": f_result,
        "evidence_ok": e_result,
        "ids_ok": not id_errors,
        "reviewer_note": reviewer_note.value.strip(),
    }
    checks = [
        q_result,
        f_result,
        e_result,
        not id_errors,
    ]

    if all(value is True for value in checks):
        row["status"] = "READY"
    elif any(value is False for value in checks):
        row["status"] = "REJECTED"
    else:
        row["status"] = "PENDING_REVIEW"

    row["reference"] = make_reference(row)
    save_all()

    with status_output:
        clear_output()
        print(
            row["case_id"],
            "저장 완료",
            row["status"],
        )


def move(step):
    state["index"] = min(
        max(state["index"] + step, 0),
        len(candidates) - 1,
    )
    render()


def approve_current(_):
    question_ok.value = True
    facts_ok.value = True
    evidence_ok.value = True
    save_current()
    move(1)


def reject_current(_):
    save_current(force_reject=True)
    move(1)


previous_button.on_click(lambda _: move(-1))
next_button.on_click(lambda _: move(1))
approve_button.on_click(approve_current)
save_button.on_click(lambda _: save_current())
reject_button.on_click(reject_current)

display(
    title,
    widgets.HTML("<b>질문</b>"),
    question_box,
    widgets.HTML("<b>기대 사실</b>"),
    facts_box,
    widgets.HTML("<b>자동 생성 reference</b>"),
    reference_box,
    widgets.HTML("<b>그래프 관계</b>"),
    graph_box,
    widgets.HTML("<b>근거 원문</b>"),
    evidence_box,
    id_result,
    question_ok,
    facts_ok,
    evidence_ok,
    reviewer_note,
    widgets.HBox(
        [
            previous_button,
            approve_button,
            save_button,
            reject_button,
            next_button,
        ]
    ),
    status_output,
)
render()


HTML(value='')

HTML(value='<b>질문</b>')

Textarea(value='', disabled=True, layout=Layout(height='90px', width='100%'))

HTML(value='<b>기대 사실</b>')

Textarea(value='', disabled=True, layout=Layout(height='140px', width='100%'))

HTML(value='<b>자동 생성 reference</b>')

... (7 outputs left out)

ToggleButtons(description='기대 사실', options=(('미검수', None), ('통과', True), ('수정·제외', False)), value=None)

ToggleButtons(description='근거', options=(('미검수', None), ('통과', True), ('수정·제외', False)), value=None)

Textarea(value='', layout=Layout(height='80px', width='100%'), placeholder='수정 사항이나 제외 이유')

Output()

## 최종 골든셋 생성

모든 문항의 검수가 끝난 뒤 실행합니다. READY 문항만 GS001부터 다시 번호를 부여하며, reference는 최종 기대 사실에서 다시 생성합니다.


In [21]:
reviewed_candidates = read_jsonl(REVIEWED_PATH)

unfinished_ids = [
    row["case_id"]
    for row in reviewed_candidates
    if row.get("status") not in {"READY", "REJECTED"}
]
if unfinished_ids:
    raise ValueError(
        f"아직 검수하지 않은 문항이 있습니다: "
        f"{unfinished_ids}"
    )

approved_candidates = [
    row
    for row in reviewed_candidates
    if row.get("status") == "READY"
]
rejected_candidates = [
    row
    for row in reviewed_candidates
    if row.get("status") == "REJECTED"
]

FINAL_KEYS = [
    "case_id",
    "question",
    "reference",
    "question_type",
    "difficulty",
    "query_strategy",
    "expected_behavior",
    "expected_facts",
    "expected_entity_ids",
    "expected_relation_ids",
    "expected_evidence_ids",
    "expected_source_chunk_ids",
    "safety_assertions",
    "status",
]
REQUIRED_KEYS = {
    "case_id",
    "question",
    "reference",
    "query_strategy",
    "expected_behavior",
    "expected_facts",
    "expected_entity_ids",
    "expected_relation_ids",
    "expected_evidence_ids",
    "expected_source_chunk_ids",
    "safety_assertions",
    "status",
}

final_golden = []
for number, candidate in enumerate(
    approved_candidates,
    start=1,
):
    source = copy.deepcopy(candidate)
    source["reference"] = make_reference(source)
    row = {
        key: source[key]
        for key in FINAL_KEYS
        if key in source
    }
    row["case_id"] = f"GS{number:03d}"
    row["status"] = "READY"
    final_golden.append(row)

question_counts = Counter(
    row["question"].strip()
    for row in final_golden
)
duplicate_questions = sorted(
    question
    for question, count in question_counts.items()
    if count > 1
)
if duplicate_questions:
    raise ValueError(
        f"중복 질문이 있습니다: {duplicate_questions}"
    )

for row in final_golden:
    missing_keys = REQUIRED_KEYS - set(row)
    if missing_keys:
        raise ValueError(
            f"{row['case_id']} 필수 키 누락: "
            f"{sorted(missing_keys)}"
        )
    if not row["question"].strip():
        raise ValueError(
            f"{row['case_id']} 질문이 비어 있습니다."
        )
    if not row["expected_facts"]:
        raise ValueError(
            f"{row['case_id']} 기대 사실이 비어 있습니다."
        )
    if not row["reference"].strip():
        raise ValueError(
            f"{row['case_id']} reference가 비어 있습니다."
        )
    if (
        row["expected_behavior"]
        in {"ANSWER", "ANSWER_WITH_WARNING"}
        and not row["expected_evidence_ids"]
    ):
        raise ValueError(
            f"{row['case_id']} 근거 ID가 비어 있습니다."
        )

write_jsonl(
    FINAL_PATH,
    final_golden,
    overwrite=OVERWRITE_FINAL,
)

print("전체 검수 문항:", len(reviewed_candidates))
print("검수 통과 문항:", len(final_golden))
print("제외 문항:", len(rejected_candidates))
print("최종 파일:", FINAL_PATH)
print(
    "reference 예시:",
    final_golden[0]["reference"]
    if final_golden
    else "없음",
)


전체 검수 문항: 30
검수 통과 문항: 23
제외 문항: 7
최종 파일: c:\Users\Playdata\Desktop\mle-01-p3-team2\data\evaluation\golden_set_final.jsonl
reference 예시: 이 병은 생육 후기에 많이 발생한다. 4~5월경 강우일수가 많고 다습한 환경이 계속되면 심하게 발생한다. 건전한 종자를 선별하고 소독하여 파종해야 한다. 병에 잘 걸리지 않는 품종을 선택해야 한다. 생육 중 비료가 부족하지 않도록 균형시비해야 한다.
